## Reading, Preparing, Cleaning, & Scaling Data

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.metrics import geometric_mean_score

# Reading data
train_data = pd.read_csv('Dataset-train-vf.csv')
test_data = pd.read_csv('Dataset-test-vf.csv')

# Label ncoding
train_data['y'] = train_data['y'].map({'Low': 0, 'High': 1})
test_data['y'] = test_data['y'].map({'Low': 0, 'High': 1})
train_data['x14'] = train_data['x14'].map({'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4})
test_data['x14'] = test_data['x14'].map({'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4})

# Missing values
train_data.drop(columns=['x2'], inplace=True)
train_data['x6'] = train_data['x6'].fillna(train_data['x6'].mean())
test_data.drop(columns=['x2'], inplace=True)
test_data['x6'] = test_data['x6'].fillna(train_data['x6'].mean())

# Splitting
features = train_data.iloc[:,1:-1] # x1, x2, x3, ..., x14
label = train_data.iloc[:,-1] # y
X_train, X_val, y_train, y_val = train_test_split(features, label, test_size=0.2, random_state=42)

X_test = test_data.iloc[:,1:-1]
y_test = test_data.iloc[:,-1]

# Normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Active Learning

In [6]:
def active_learning(train_data, test_data, strategy):
  X = train_data.drop(columns=['sample','y'])
  y = train_data['y']

  X_test = test_data.iloc[:,1:-1]
  y_test = test_data.iloc[:,-1]

  X_train, X_pool, y_train, y_pool = train_test_split(X, y, test_size=0.9, random_state=42)

  initial_indices = np.random.choice(X_pool.index, size=50, replace=False)
  X_train = X_pool.loc[initial_indices]
  y_train = y_pool.loc[initial_indices]

  X_pool = X_pool.drop(index=initial_indices)
  y_pool = y_pool.drop(index=initial_indices)

  X_pool, X_val, y_pool, y_val = train_test_split(X_pool, y_pool, test_size=0.2, random_state=42)

  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_pool = scaler.transform(X_pool)
  X_val = scaler.transform(X_val)
  X_test = scaler.transform(X_test)

  model = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)

  for iteration in range(10):
      print(f"Iteration {iteration + 1}")

      model.fit(X_train, y_train)

      if X_pool.shape[0] > 0:
          probs = model.predict_proba(X_pool)

          n_samples = min(10, len(X_pool))

          if strategy == "entropy":
            uncertain_indices = entropy_sampling(probs, n_samples)
          else:
            uncertain_indices = least_confidence_sampling(probs, n_samples)

          X_selected = X_pool[uncertain_indices]
          y_selected = y_pool.iloc[uncertain_indices]

          X_train = np.vstack([X_train, X_selected])
          y_train = pd.concat([y_train, y_selected], axis=0)

          X_pool = np.delete(X_pool, uncertain_indices, axis=0)
          y_pool = y_pool.drop(y_pool.index[uncertain_indices])

      if X_train.shape[0] > 0:
          y_pred = model.predict(X_val)
          accuracy = accuracy_score(y_val, y_pred)
          print(f"Validation Accuracy: {accuracy:.4f}")

  print("Active learning complete.")

  print("\nTesting")

  # Prediction
  y_pred = model.predict(X_test)

  # Evaluating
  accuracy = accuracy_score(y_test, y_pred)
  conf_matrix = confusion_matrix(y_test, y_pred)
  report = classification_report(y_test, y_pred)
  gmean_test = geometric_mean_score(y_test, y_pred, average='macro')

  print("Accuracy:", accuracy)
  print("Confusion Matrix:\n", conf_matrix)
  print("Classification Report:\n", report)
  print("GMean (Test):", gmean_test)  # Display GMean



In [7]:
# least confidence
def least_confidence_sampling(probs, n_samples):
    confidence = np.max(probs, axis=1)
    least_confident_indices = np.argsort(confidence)[:n_samples]
    return least_confident_indices

# entropy
def entropy_sampling(probs, n_samples):
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
    high_entropy_indices = np.argsort(-entropy)[:n_samples]
    return high_entropy_indices

In [8]:
strategy_entropy = "entropy"
strategy_least_confidence = "least confidence"
print("strategy_entropy\n")
active_learning_with_entropy = active_learning(train_data, test_data, strategy_entropy)

print("strategy_least_confidence\n")
active_learning_with_strategy_least_confidence = active_learning(train_data, test_data, strategy_least_confidence)

strategy_entropy

Iteration 1
Validation Accuracy: 0.8118
Iteration 2
Validation Accuracy: 0.8353
Iteration 3
Validation Accuracy: 0.8471
Iteration 4
Validation Accuracy: 0.8588
Iteration 5
Validation Accuracy: 0.8765
Iteration 6
Validation Accuracy: 0.8588
Iteration 7
Validation Accuracy: 0.8647
Iteration 8
Validation Accuracy: 0.8706
Iteration 9
Validation Accuracy: 0.8706
Iteration 10
Validation Accuracy: 0.8706
Active learning complete.

Testing
Accuracy: 0.8938271604938272
Confusion Matrix:
 [[245   7]
 [ 36 117]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.97      0.92       252
           1       0.94      0.76      0.84       153

    accuracy                           0.89       405
   macro avg       0.91      0.87      0.88       405
weighted avg       0.90      0.89      0.89       405

GMean (Test): 0.8684640522875817
strategy_least_confidence

Iteration 1
Validation Accuracy: 0.8765
Iteration 2
Validation A

Active learning demonstrates strong performance; however, Adaboost continues to outperform it.